# Cell Migration RTM Pipeline
This notebook showcases an example how to do directed migration as shown in the real-time feedback microscopy paper. The frontal part of the cells are stimulated using a DMD. This Demo runs through the MicroManager demo microscope with data from an old experiment. To see how it works on a real microscope, please see 02_CellMigration_Mic.ipynb. 

### Import required libraries

In [ ]:
import time
from faro.core.data_structures import Channel, StimTreatment, SegmentationMethod
import faro.core.utils as utils

### Experimental Settings

In [ ]:
from faro.microscope.Inscoper import InscoperMicroscope
from inscoper_api import SubDeviceId

# INSCOPER_CONFIG_FOLDER   = r"C:\Users\rtorro\Documents\Utils\inscoper_useq\configs\configXYZ"
# INSCOPER_CHANNELS_FOLDER = r"C:\Users\rtorro\Documents\IIS\channels"
# INSCOPER_MAIN_CAMERA     = "CameraVirtual"
# INSCOPER_CHANNEL_GROUP   = "Channel"

# ─── Inscoper configuration ────────────────────────────────────────────
# Point these to YOUR Inscoper setup
# INSCOPER_CONFIG_FOLDER   = r"C:\Users\rtorro\Documents\Utils\inscoper_useq\configs\configXYZ"
# INSCOPER_CHANNELS_FOLDER = r"C:\Users\rtorro\Documents\IIS\channels"
# INSCOPER_MAIN_CAMERA     = "CameraVirtual"
# INSCOPER_CHANNEL_GROUP   = "Channel"          # adjust to your channel group
# # # ──────────────────────────────────────────────────────────────────────
INSCOPER_CONFIG_FOLDER   = r"C:\Users\InscoperPasDev\Desktop\\Remy\InscoperInterface-9.3.10\SystemConfigurations\FullSystemWithHamamatuNoLumencor"
INSCOPER_CHANNELS_FOLDER = r"C:\Users\InscoperPasDev\Desktop\Remy\InscoperInterface-9.3.10\channels"
INSCOPER_MAIN_CAMERA     = "Fusion_Right"
INSCOPER_CHANNEL_GROUP   = "Channel"          # adjust to your channel group
# # ──────────────────────────────────────────────────────────────────────

mic = InscoperMicroscope(
    config_folder=INSCOPER_CONFIG_FOLDER,
    channels_folder=INSCOPER_CHANNELS_FOLDER,
    main_camera=INSCOPER_MAIN_CAMERA,
    channel_group=INSCOPER_CHANNEL_GROUP,
)

# mic.mmc.setValue(SubDeviceId("CameraVirtual", "ImageHeight"), 2048)
# mic.mmc.setValue(SubDeviceId("CameraVirtual", "ImageWidth"), 2048)

#
# mic.mmc.setValue(
#     SubDeviceId("CameraTif", "Directory"),
#     r"C:\Users\rtorro\Documents\Utils\VirtualCameraNuclei\hela",
# )
mic.mmc.setValue(SubDeviceId('Hamamatsu_Data', "SUBARRAY HPOS1"), "0")
mic.mmc.setValue(SubDeviceId('Hamamatsu_Data', "SUBARRAY VPOS1"), "0")
mic.mmc.setValue(SubDeviceId('Hamamatsu_Data', "SUBARRAY HSIZE1"), "512")
mic.mmc.setValue(SubDeviceId('Hamamatsu_Data', "SUBARRAY VSIZE1"), "512")
mic.mmc.setValue(SubDeviceId('Hamamatsu_Data', "SUBARRAY MODE1"), "ON")

# from faro.microscope.MMDemo import MMDemo

# path_with_old_data_for_simulation = os.path.join(".", "test_exp_data", "02_demo_imgs")

# mic = MMDemo(old_data_project_path=path_with_old_data_for_simulation)

In [ ]:
import os

## Configuration options - set experiment timing, storage and stimulation parameters
# General timing and frame counts:
N_FRAMES = 30 * 4  # number of timesteps
# If you want the notebook/script to wait before starting the experiment, set this (hours).
# Useful for scheduling: set to a fractional value (e.g. 0.5 for 30 minutes).
SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

# Timing for acquisition: interval between timesteps (seconds) and approximate time per FOV (seconds).
# TIME_BETWEEN_TIMESTEPS: time between frames (across all FOVs).
TIME_BETWEEN_TIMESTEPS = 15  # seconds between timesteps
# TIME_PER_FOV: approximate time required to image one FOV (used for scheduling/estimates).
TIME_PER_FOV = 3.75  # seconds per FOV (camera + stage moves + overhead)

# Display / bookkeeping options:
# If True, add a column/group that stores the last stimulation exposure applied to each FOV (helpful for QC).
ADD_STIM_EXPOSURE_GROUP = False  # set to True to save per-FOV last-stim-exposure info
# When True, stim timepoints will be distributed evenly across the available timesteps rather than using explicit lists.
REGULAR_SPACING_BETWEEN_STIMULATIONS = False  # True -> evenly spaced stim timings; False -> use explicit stim_timestep lists

## Storage path for the experiment - change to your desired directory and experiment name.
base_path = "C:\\Users\\InscoperPasDev\\Desktop\\Remy\\exps"  # example: 'E:\Alex' (double-escaped for JSON/Notebook)
experiment_name = "testmigration"
path = os.path.join(base_path, experiment_name)

## Channels: images to acquire each timestep.
# Each Channel(...) maps to a microscope channel name configured in Micro-Manager/your device.
# If exposure or power is omitted, the hardware default (set in the device/GUI) will be used.
# Examples: specify different exposures per channel, or add channels without exposures to use defaults.
channels = []
channels.append(Channel(config="TL", exposure=30))  # example: ERK-KTR reporter

# Optional channel to run an optocheck (check expression of optogenetic marker).
# If the optogenetic marker is imaged in the same channel as stimulation, you can reuse that channel here.
channel_optocheck = Channel(
    config="TL", exposure=50
)  # high exposure for low-signal marker
optocheck_timepoints = (
    N_FRAMES - 1,
)  # tuple of timesteps at which to capture the optocheck channel

# Experimental condition(s): a list of labels assigned to FOVs.
# You can repeat or expand this list to match the number of FOVs.
condition = [
    "optoTIAM_single",
]
# Example alternatives (uncomment to use):
# - Repeat each condition multiple times: condition = [cond for cond in condition for _ in range(repeats)]
# - Create a long condition vector: condition = ["optoFGFR_high"] * 24 + ["optoFGFR"] * 24

# If using wellplates, set how many FOVs per well. Set to None if not using wellplates.
n_fovs_per_well = None  ## number of FOVs per well; use None for free-FOV experiments

# Stimulation parameters for optogenetics. Define a list of StimTreatment objects per phase.
# Notes on StimTreatment fields:
# - treatment_name: human-readable label for the treatment
# - stim_timestep: a tuple/list of integers (timesteps) when stimulation should occur, OR a range/tuple.
#       Examples: (10,20,30), list(range(10,100)), (tuple(range(10,100,1)))
# - stim_exposure_list: either a single exposure value (applied to all listed timesteps) or a list/tuple of exposures matching the timesteps.
# - auto_repeat_stim_exposure: when True and a single exposure value is provided, it will be repeated across timesteps.
# - stim_power/stim_channel_name/stim_channel_group: map the stimulation settings to your device/channel group.
# - stim_channel_device_name/stim_channel_power_property_name: lower-level device settings used by the hardware API.

stim_phase = [
    StimTreatment(
        treatment_name="15min_stim",
        stim_timestep=(tuple(range(1, 30 * 4, 1))),  # example: stim at timesteps 10..99
        stim_exposure_list=200,
        stim_power=100,
        stim_channel_name="TL",
        stim_channel_group="Channel",
        stim_channel_device_name="NikonTi2",
        stim_channel_power_property_name="TL Lamp Intensity (%)",
        auto_repeat_stim_exposure=True,
    )
]

# a list of StimTreatment objects; when multiple are supplied they will be assigned across FOVs according to the utils.apply_stim_treatments_to_df_acquire logic

# # Print the final stim schedule for confirmation before running. Helpful to verify exposures/timings.
# for stim_phase in [
#     stim_phase,
# ]:
#     utils.print_stim_exposures_timesteps(stim_phase)

## Define the Tools that you are using for the experiment (segmentors, trackers, feature extractors, stimulator).
from faro.stimulation.percentage_of_cell import StimPercentageOfCell
from faro.stimulation.base import StimWholeFOV
from faro.tracking.trackpy import TrackerTrackpy
from faro.feature_extraction.simple import SimpleFE
from faro.feature_extraction.ref import RefFE
from faro.segmentation.threshold import SegmentatorThreshold

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=SegmentatorThreshold(threshold=0.1),
        use_channel=0,
        save_tracked=True,
    )
]

stimulator = StimPercentageOfCell()
feature_extractor = SimpleFE("labels")
tracker = TrackerTrackpy()
optocheck = RefFE(used_mask="labels")


from faro.core.pipeline import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_ref=optocheck,
)

### GUI - Napari Micromanager

#### Load GUI

In [ ]:
### Base GUI ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

In [ ]:
# The calibration light path is now passed per call rather than being
# fixed on the microscope. power=100 matches the old
# DMD_CALIBRATION_PROFILE (NikonTi2 / TL Lamp Intensity (%)).
from faro.core.data_structures import PowerChannel

calibration_channel = PowerChannel(config="TL", exposure=25, power=100)
mic.calibrate_dmd(calibration_channel)

### Map Experiment to FOVs

### Use FOVs to generate dataframe for acquisition

In [ ]:
fovs = utils.generate_fov_objects(mic, viewer=viewer)


df_acquire = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    condition=condition,
)
df_acquire = utils.apply_stim_treatments_to_df_acquire(
    df_acquire,
    stim_phase,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire

To show that df_acquire contains all required experiment data, and that the stimulation module *percentage_of_cell* can also access the metadata, we modify here the stimulation condition for the second timestep and stimulate 40% of the cell instead of 30%. 


In [ ]:
df_acquire["stim_cell_percentage"] = 0.3
df_acquire.loc[1, "stim_cell_percentage"] = 0.4
df_acquire

### Run experiment

In [ ]:
from faro.core.controller import Controller
from faro.core.conversion import df_to_events

for _ in range(0, SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600):
    time.sleep(1)

try:
    mm_wdg._core_link.cleanup()

except:
    pass

events = df_to_events(df_acquire)
print("events ready")

ctrl = Controller(mic, pipeline)
print("controller initialized")

ctrl.run_experiment(events, stim_mode="current").wait()
print("run experiment done")

mic.post_experiment()
print("post experiment done")

time.sleep(10)

utils.generate_exp_data_from_tracks(path)

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)